# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:54<00:00, 18.22s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

'Title: Refurb Acer Chromebook Plus 14" Touchscreen Laptop for $149 + free shipping\nDetails: This is the lowest price we\'ve ever seen for this budget-friendly Chromebook Plus. A new model would set you back $340 at Best Buy. A 1-year Allstate warranty applies.cost Buy Now at eBay\nFeatures: Intel Core i3-N305 Alder Lake-N processor 14" touchscreen IPS display 8GB RAM, 512GB SSD ChromeOS Model: CB514-4HT\nURL: https://www.dealnews.com/products/Acer/Acer-Chromebook-Plus-14-Touchscreen-Laptop/491306.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: LG Daily Deals at Best Buy: Up to 50% off + free shipping
Details: For one day only, Best Buy is knocking up to 50% off LG TVs, ranges, audio equipment, and a monitor and laptop. Pictured is the best deal of the bunch: the LG B5 Series OLED77B5PUA 77" 4K HDR OLED UHD Smart TV, which is $1,500 (half its original price). Shop Now at Best Buy
Features: 
URL: https://www.dealnews.c

In [ ]:
# Expected Response format from LLM
DealSelection.model_json_schema()

{'$defs': {'Deal': {'description': 'A class to Represent a Deal with a summary description',
   'properties': {'product_description': {'description': "Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose.",
     'title': 'Product Description',
     'type': 'string'},
    'price': {'description': 'The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200',
     'title': 'Price',
     'type': 'number'},
    'url': {'description': 'The URL of the deal, as provided in the input',
     'title': 'Url',
     'type': 'string'}},
   'required': ['product_description', 'price', 'url'],
   'title': 'Deal',
   'type': 'object'}},
 'description': 'A clas

In [9]:
# For Structured Output, we need to specify the response_format and .parse() method
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='LG B5 Series OLED77B5PUA is a 77-inch 4K HDR OLED UHD Smart TV offering OLED picture quality with deep blacks and wide viewing angles. It runs LG’s smart TV platform for streaming apps and smart features, suitable for home theater use given its large diagonal and OLED panel advantages. The model is positioned as a high-end display for movies, gaming, and general media consumption.', price=1500.0, url='https://www.dealnews.com/LG-Daily-Deals-at-Best-Buy-Up-to-50-off-free-shipping/21801723.html?iref=rss-c142'), Deal(product_description='Samsung QN990F is an 8K Vision AI Smart TV available in multiple sizes; the 65-inch model delivers 8K resolution with Samsung’s Vision AI processing for upscaling and enhanced picture refinement. It includes smart TV features and modern connectivity for UHD sources and streaming, aimed at viewers seeking the highest-resolution panel and advanced image processing.', price=5299.99, url='https://www.dealnews.com

In [12]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


LG B5 Series OLED77B5PUA is a 77-inch 4K HDR OLED UHD Smart TV offering OLED picture quality with deep blacks and wide viewing angles. It runs LG’s smart TV platform for streaming apps and smart features, suitable for home theater use given its large diagonal and OLED panel advantages. The model is positioned as a high-end display for movies, gaming, and general media consumption.
1500.0
https://www.dealnews.com/LG-Daily-Deals-at-Best-Buy-Up-to-50-off-free-shipping/21801723.html?iref=rss-c142

Samsung QN990F is an 8K Vision AI Smart TV available in multiple sizes; the 65-inch model delivers 8K resolution with Samsung’s Vision AI processing for upscaling and enhanced picture refinement. It includes smart TV features and modern connectivity for UHD sources and streaming, aimed at viewers seeking the highest-resolution panel and advanced image processing.
5299.99
https://www.dealnews.com/Samsung-QN990-F-8-K-Samsung-Vision-AI-Smart-TV-Deals-Up-to-500-off-free-shipping/21801640.html?iref=rs

In [13]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [16]:
result

DealSelection(deals=[Deal(product_description='LG B5 Series OLED77B5PUA is a 77-inch 4K HDR OLED UHD smart television that delivers self-emissive OLED contrast with deep blacks and wide viewing angles. It includes smart TV features for streaming apps and likely supports modern HDMI inputs and HDR formats for enhanced picture quality. The model is positioned as a large, premium display suitable for home theaters and gaming.', price=1500.0, url='https://www.dealnews.com/LG-Daily-Deals-at-Best-Buy-Up-to-50-off-free-shipping/21801723.html?iref=rss-c142'), Deal(product_description="Samsung QN990F is an 8K Vision AI smart TV available in multiple sizes; the 65-inch model offers an ultra-high 8K resolution panel with Samsung's AI upscaling and smart TV platform for apps and streaming. It targets viewers who want maximum resolution, advanced processing for upscaling, and premium display performance for large-format rooms.", price=5299.99, url='https://www.dealnews.com/Samsung-QN990-F-8-K-Samsu

In [17]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

LG B5 Series OLED77B5PUA is a 77-inch 4K HDR OLED UHD Smart TV offering OLED picture quality with deep blacks and wide viewing angles. It runs LG’s smart TV platform for streaming apps and smart features, suitable for home theater use given its large diagonal and OLED panel advantages. The model is positioned as a high-end display for movies, gaming, and general media consumption.
1500.0
https://www.dealnews.com/LG-Daily-Deals-at-Best-Buy-Up-to-50-off-free-shipping/21801723.html?iref=rss-c142

Samsung QN990F is an 8K Vision AI Smart TV available in multiple sizes; the 65-inch model delivers 8K resolution with Samsung’s Vision AI processing for upscaling and enhanced picture refinement. It includes smart TV features and modern connectivity for UHD sources and streaming, aimed at viewers seeking the highest-resolution panel and advanced image processing.
5299.99
https://www.dealnews.com/Samsung-QN990-F-8-K-Samsung-Vision-AI-Smart-TV-Deals-Up-to-500-off-free-shipping/21801640.html?iref=rs

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")